# SetuHQ quick start: the stored pi Men run, offline

<!-- SPDX-License-Identifier: AGPL-3.0-only -->

This notebook walks through the run SetuHQ keeps in the repository, `runs/tess-pi-men-s1`: TESS sector 1 of
pi Men (TIC 261136679), the star of the known planet pi Men c (Huang et al. 2018). Everything here reads
committed files and runs SetuHQ's own code; **nothing is downloaded and no AI provider is called.**

1. load the run's candidates,
2. look at the data behind the Docket (the candidate case file),
3. run a Probe (a deterministic matched-filter reviewer) on a stored light curve,
4. call an MCP tool as a plain Python function.

**Every SetuHQ output is a candidate, never a confirmation.** The dips below are candidates that a reviewer
panel marked and the pipeline vetted; whether a dip is a planet is decided by follow-up observations, and
for pi Men c it was decided long before SetuHQ looked.

Run it from a checkout of https://github.com/ashfordeOU/SetuHQ with `pip install -e ".[mcp]"`. The numbers
it prints come from the stored run, so they follow whatever that run holds.

In [ ]:
# Find the checkout that holds the stored run, and point SetuHQ's data root at it.
# In Google Colab (no checkout), clone the repository and install it first; that cell needs the network.
import os
import sys
from pathlib import Path

RUN = "runs/tess-pi-men-s1"


def find_root(start: Path) -> Path | None:
    for d in (start, *start.parents):
        if (d / RUN / "manifest.json").is_file():
            return d
    return None


root = find_root(Path.cwd().resolve())
if root is None and "google.colab" in sys.modules:
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ashfordeOU/SetuHQ"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "SetuHQ[mcp]"], check=True)
    root = Path("SetuHQ").resolve()
    sys.path.insert(0, str(root / "src"))  # this kernel started before the install
if root is None:
    raise SystemExit(f"run this notebook inside a SetuHQ checkout (no {RUN} above {Path.cwd()})")

os.environ["SETUHQ_ROOT"] = str(root)  # the MCP tools read and write only inside it
os.chdir(root)

import setuhq

print("SetuHQ", setuhq.__version__, "| data root:", root)

## 1. The candidates

`setuhq.tools` holds the functions the MCP server exposes. Each returns an envelope: the `result`, its
`provenance` (code version, the stored files read and their SHA-256) and `honesty` labels.

In [ ]:
from setuhq import tools

listing = tools.list_candidates(RUN)
cands = listing["result"]["candidates"]
print(f"{listing['result']['n']} candidates in {listing['result']['run']}\n")
print(f"{'candidate':<24}{'passed':>7}{'depth ppm':>11}{'SNR':>7}{'period d':>10}  FAP")
for c in cands:
    period = f"{c['period_days']:.3f}" if c.get("period_days") else "-"
    print(f"{c['candidate_id']:<24}{str(c['passed']):>7}{c['depth_ppm']:>11.0f}{c['snr']:>7.1f}{period:>10}  {c.get('fap_label') or '-'}")

print("\nhonesty:", *listing["honesty"], sep="\n  ")

## 2. The Docket's data

The Docket (https://setuhq.com/docket/) is built from `docket_data`: every candidate with its vetting
checks and outcome, the signals they group into, every reviewer's marks and the cross-match with the NASA
Exoplanet Archive stored beside the run.

In [ ]:
from setuhq.docket import docket_data

docket = docket_data(RUN, title="pi Men (TESS sector 1)")
print("counts:", docket["counts"])
print("reviewers:", ", ".join(docket["reviewer_names"][r]["name"] for r in docket["reviewers"]))
print()
for c in docket["candidates"]:
    states = ", ".join(f"{ch['key']}={ch['state']}" for ch in c["checks"])
    print(f"{c['candidate_id']:<24} {c['outcome']:<28} {states}")
print()
print(len(docket["signals"]), "signal(s)")
for s in docket["signals"]:
    period = f"{s['period_days']:.4f} d" if s['period_days'] else "single event"
    print(f"  {s['signal_id']}: {period}, {len(s['dips'])} dip(s), outcome {s['outcome']}")

## 3. A Probe on a stored light curve

The run keeps every blind-coded subject it reviewed (`subjects/*.json`: time, flux and flux error of one
13-day segment, with an opaque id). A Probe is a `SimulatedVolunteer`: a single-event matched filter plus a
box least-squares search, seeded, with a stated miss probability and boundary jitter. This is the
*Typical* Probe of the run's config (`configs/tess_pi_men_s1.toml`).

In [ ]:
import json

import numpy as np

from setuhq.reviewers.simulated import SimulatedVolunteer
from setuhq.subjects import Subject

path = sorted((root / RUN / "subjects").glob("*.json"))[0]
raw = json.loads(path.read_text())
subject = Subject(subject_id=raw["subject_id"], mission="TESS", time_unit=raw["time_unit"],
                  time=np.asarray(raw["time"]), flux=np.asarray(raw["flux"]), flux_err=np.asarray(raw["flux_err"]))
print(f"subject {subject.subject_id}: {len(subject)} points, {subject.start:.2f} to {subject.end:.2f} {subject.time_unit}")

probe = SimulatedVolunteer("volunteer-typical", snr_threshold=5.0, miss_probability=0.10, seed=12)
verdict = probe.review(subject)
print(f"label: {verdict.label}; {len(verdict.marks)} mark(s)")
for m in verdict.marks:
    print(f"  dip from {m.t_start:.3f} to {m.t_end:.3f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(subject.time, (subject.flux - 1) * 1e6, ".", ms=2, color="0.3")
for m in verdict.marks:
    ax.axvspan(m.t_start, m.t_end, color="tab:orange", alpha=0.3)
ax.set_xlabel(f"time ({subject.time_unit})")
ax.set_ylabel("flux - 1 (ppm)")
ax.set_title(f"{subject.subject_id}: the Typical Probe's marks (candidates, not confirmations)")
fig.tight_layout()

## 4. An MCP tool, called as a function

`setuhq-mcp` registers these same functions as MCP tools; an assistant gets exactly what you get here.
`get_candidate` returns one candidate with its fit and every reviewer's verdict; `transit_depth` is one of
the Methods physics tools.

In [ ]:
one = tools.get_candidate(cands[0]["candidate_id"], RUN)
cand = one["result"]["candidate"]
print("candidate:", cands[0]["candidate_id"], "| passed:", cand["passed"], "| depth ppm:", round(cand["depth_ppm"]))
print("provenance keys:", sorted(one["provenance"]))

depth = tools.transit_depth(radius_ratio=0.0155)
print("geometric depth for Rp/R* = 0.0155:", round(depth["result"]["geometric_depth_ppm"]), "ppm")
print("tools the server exposes:", ", ".join(fn.__name__ for fn, _ in tools.TOOLS))

## Next

- `setuhq rebuild runs/tess-pi-men-s1 configs/tess_pi_men_s1.toml` re-vets the run from its stored
  classifications; `setuhq docket runs/tess-pi-men-s1 --out site/docket --title ...` builds the Docket page.
- `setuhq-mcp` serves these tools to Claude Desktop, Claude Code, Cursor and others: `docs/MCP-INSTALL.md`.
- The benchmark, the leaderboard and their limits are in the README's Evidence section.